# 03 - Synthetic experts and scoring test

**Owner:** Daria with Valeri  |  **Output:** `data/synthetic/` (experts.csv, responses.csv, scoreboard.csv, panel.csv)

**Everything about the experts here is simulated.** The point is to test the scoring pipeline on real outcomes: if better simulated experts get better scores, the scoreboard idea works. It says nothing about how real experts will perform.

Score used: the Brier score, the squared gap between a forecast (0 to 1) and what happened (1 = materialised, 0 = not). Lower is better. Always guessing 50% scores 0.25.

## 0. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Find the repo folder (the one that contains src/risk_tools.py), wherever this notebook is run from
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "risk_tools.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not find src/risk_tools.py above {Path.cwd()}. Put risk_tools.py in the src/ folder of the repo.")
sys.path.append(str(ROOT / "src"))
from risk_tools import *
print("Repo folder:", ROOT)


Repo folder: c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth


In [4]:
PROCESSED   = ROOT / "data" / "processed"
OUT_DIR     = ROOT / "data" / "synthetic"
N_EXPERTS   = 10
ANSWER_RATE = 0.6    # share of risks each expert answers
MIN_ANSWERS = 10     # minimum answers before an expert appears on the scoreboard
SEED        = 7

## 1. Baselines to beat

In [5]:
resolved = pd.read_csv(PROCESSED / "resolved_risks.csv")
prior = resolved["acaps_p"].fillna(0.5)
base_rate = resolved["outcome"].mean()

baselines = pd.Series({
    "Always 50%": brier(np.full(len(resolved), 0.5), resolved["outcome"]).mean(),
    "Always the average rate": brier(np.full(len(resolved), base_rate), resolved["outcome"]).mean(),
    "ACAPS rating as a forecast": brier(prior, resolved["outcome"]).mean(),
}).round(3)
print(f"{len(resolved)} resolved risks, {base_rate:.0%} materialised")
baselines.to_frame("mean Brier")

104 resolved risks, 59% materialised


,mean Brier
Always 50%,0.250
Always the average rate,0.243
ACAPS rating as a forecast,0.215


## 2. Simulate the panel and its answers

In [6]:
experts = simulate_experts(N_EXPERTS, seed=SEED)
responses = simulate_responses(resolved, experts, prior_col="acaps_p", answer_rate=ANSWER_RATE, seed=SEED + 4)
print(len(responses), "answers from", responses["expert_id"].nunique(), "experts")
experts

627 answers from 10 experts


,expert_id,pseudonym,anonymity_level,sim_skill,sim_bias,sim_noise
0,E01,Falcon,known_to_acaps,0.16,-0.14,0.66
1,E02,Kestrel,known_to_acaps,0.61,-0.29,0.89
2,E03,Sparrow,pseudonymous,0.04,-0.24,0.75
3,E04,Marten,pseudonymous,0.04,0.32,0.82
4,E05,Acacia,pseudonymous,0.51,-0.24,0.36
5,E06,Linden,anonymous,0.47,-0.01,0.68
6,E07,Heron,anonymous,0.92,0.27,0.66
7,E08,Cedar,pseudonymous,0.63,-0.18,0.91
8,E09,Juniper,anonymous,0.51,-0.03,0.55
9,E10,Badger,known_to_acaps,0.50,0.03,0.72


## 3. Scoreboard

In [7]:
board = scoreboard(responses, experts, MIN_ANSWERS)
board

,rank,pseudonym,answers,mean_brier,expert_id
0,1,Heron,53,0.050,E07
1,2,Kestrel,66,0.070,E02
2,3,Juniper,62,0.074,E09
3,4,Linden,61,0.082,E06
4,5,Acacia,72,0.086,E05
5,6,Cedar,61,0.091,E08
6,7,Badger,61,0.101,E10
7,8,Falcon,66,0.161,E01
8,9,Marten,63,0.221,E04
9,10,Sparrow,62,0.245,E03


In [8]:
# Does the scoreboard recover the hidden skill? (should be clearly negative: more skill, lower Brier)
check = board.merge(experts[["expert_id", "sim_skill"]], on="expert_id")
corr = check["sim_skill"].rank().corr(check["mean_brier"].rank())
print(f"Rank correlation between hidden skill and Brier score: {corr:.2f}")

Rank correlation between hidden skill and Brier score: -0.82


## 4. Does the group beat ACAPS's own rating?
The simulated experts are built to know a bit more than the rating, so the panel scoring better is true by construction. It shows the scoring works. It does not show that real experts will beat ACAPS, so do not claim that in the pitch.

In [9]:
panel = aggregate_panel(responses)
results = pd.Series({
    "Always 50%": baselines["Always 50%"],
    "ACAPS rating as a forecast": baselines["ACAPS rating as a forecast"],
    "Panel mean": brier(panel["panel_mean"], panel["outcome"]).mean(),
    "Panel median": brier(panel["panel_median"], panel["outcome"]).mean(),
    "Best single expert": board["mean_brier"].min(),
    "Worst single expert": board["mean_brier"].max(),
}).round(3).sort_values()
results.to_frame("mean Brier")

,mean Brier
Best single expert,0.050
Panel median,0.098
Panel mean,0.102
ACAPS rating as a forecast,0.215
Worst single expert,0.245
Always 50%,0.250


## 5. Calibration: when the panel says 70%, does it happen about 70% of the time?

In [10]:
calibration_table(panel["panel_mean"], panel["outcome"])

,n,mean_forecast,observed_rate
band,,,
"(-0.001, 0.2]",13,0.15,0.08
"(0.2, 0.4]",22,0.30,0.00
"(0.4, 0.6]",16,0.48,0.56
"(0.6, 0.8]",34,0.72,0.94
"(0.8, 1.0]",19,0.87,1.00


## 6. Save

In [11]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
experts.to_csv(OUT_DIR / "experts.csv", index=False)
responses.to_csv(OUT_DIR / "responses.csv", index=False)
board.to_csv(OUT_DIR / "scoreboard.csv", index=False)
panel.to_csv(OUT_DIR / "panel.csv", index=False)
print(sorted(p.name for p in OUT_DIR.iterdir()))

['experts.csv', 'panel.csv', 'responses.csv', 'scoreboard.csv']


Notes for the team
- `experts.csv` contains the hidden simulation values (`sim_*`). The app should only use `expert_id`, `pseudonym` and `anonymity_level`.
- The scoreboard shows pseudonyms only, and identity stays separate from answers. That is the design to explain when the jury asks about anonymity.
- In the pitch, say clearly that the experts are simulated and the outcomes are real.